In [ ]:
import joblib
import numpy as np
from create_dataset import vectorize_state
from game import sample_until, sample_every, sample
import os
import json
from scipy import stats
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import random

In [ ]:
with open('item.json') as f:
    item_data = json.load(f)

In [ ]:
model = joblib.load('model.pkl')

In [ ]:
root = '../dataset/test'

all_files = []

for dirpath, dirnames, filenames in os.walk(root):
    for filename in filenames:
        all_files.append(os.path.join(dirpath, filename))

In [ ]:
def format_wpa_summary(mean, me):
    def format_percentage(number):
        return f'{round(number * 100, 2)}%'

    return f'{format_percentage(mean)} ± {format_percentage(me)}'

In [ ]:
def calc_first_event_wpa(callback):
    winrates = []
    wpes = []
    
    # Loop through all games
    for game_path in all_files:
        with open(game_path) as f:
            game = json.load(f)
        state, outcome = sample_until(game, callback)
        # outcome is a boolean that depics which team got the event, False for red team True for blue team
        if state:
            winrates.append(state['win'] == outcome)
            v = vectorize_state(state)
            v = np.array(v, dtype=np.float32)
            v = np.expand_dims(v, 0)
            prob = model.predict_proba(v)[0, outcome].item()
            wpes.append(prob)

    return np.array(winrates), np.array(wpes)

In [ ]:
def calc_average_event_wpa(callback):
    winrates = []
    wpes = []
    
    # Loop through all games
    for game_path in all_files:
        with open(game_path) as f:
            game = json.load(f)
        results = sample_every(game, callback)
        # outcome is a boolean that depics which team got the event, False for red team True for blue team
        for state, outcome in results:
            if state:
                winrates.append(state['win'] == outcome)
                v = vectorize_state(state)
                v = np.array(v, dtype=np.float32)
                v = np.expand_dims(v, 0)
                prob = model.predict_proba(v)[0, outcome].item()
                wpes.append(prob)

    return np.array(winrates), np.array(wpes)

In [ ]:
def calculate_wpa_ci(wr, we):
    d = wr - we
    n = len(d)
    mean = d.mean()
    std = d.std(ddof=1)
    sem = std / np.sqrt(n)
    t_value = stats.t.ppf(0.975, df=n-1)
    me = t_value * sem + 0.008
    return mean, me

In [ ]:
def get_item_callback(item_id):
    def item_callback(state, event):
        if event['type'] == 'ITEM_PURCHASED' and event['itemId'] == item_id:
            return True, 1 - int((event['participantId'] - 1) / 5)
        return False, None

    return item_callback

In [ ]:
def champion_kill(state, event):
    if event['type'] == 'CHAMPION_KILL':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def baron(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'BARON_NASHOR':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def elder(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'DRAGON' and event['monsterSubType'] == 'ELDER_DRAGON':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def dragon(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'DRAGON' and event['monsterSubType'] != 'ELDER_DRAGON':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def void_grubs(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'HORDE':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def riftherald(state, event):
    if event['type'] == 'ELITE_MONSTER_KILL' and event['monsterType'] == 'RIFTHERALD':
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def top_tower(state, event):
    if event['type'] == 'BUILDING_KILL' and event['buildingType'] == 'TOWER_BUILDING' and event['laneType'] == 'TOP_LANE' and event['killerId'] != 0 and sum(state['teams'][0]['towers']) == 9 and sum(state['teams'][1]['towers']) == 9:
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def mid_tower(state, event):
    if event['type'] == 'BUILDING_KILL' and event['buildingType'] == 'TOWER_BUILDING' and event['laneType'] == 'MID_LANE' and event['killerId'] != 0 and sum(state['teams'][0]['towers']) == 9 and sum(state['teams'][1]['towers']) == 9:
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
def bot_tower(state, event):
    if event['type'] == 'BUILDING_KILL' and event['buildingType'] == 'TOWER_BUILDING' and event['laneType'] == 'BOT_LANE' and event['killerId'] != 0 and sum(state['teams'][0]['towers']) == 9 and sum(state['teams'][1]['towers']) == 9:
        return True, 1 - int((event['killerId'] - 1) / 5)
    return False, None

In [ ]:
item_ids = [3363, 3157, 3916, 3165]

for item_id in item_ids:
    wr, we = calc_event_wpa(get_item_callback(item_id))
    mean, me = calculate_wpa_ci(wr, we)
    item_name = item_data['data'][str(item_id)]['name']
    print(f'{item_name}: ' + format_wpa_summary(mean, me))

In [ ]:
item_ids = [2031, 2055, 1082, 3364]

for item_id in item_ids:
    wr, we = calc_event_wpa(get_item_callback(item_id))
    mean, me = calculate_wpa_ci(wr, we)
    item_name = item_data['data'][str(item_id)]['name']
    print(f'{item_name}: ' + format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_average_event_wpa(void_grubs)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_first_event_wpa(riftherald)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_average_event_wpa(baron)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_average_event_wpa(elder)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_average_event_wpa(dragon)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_first_event_wpa(champion_kill)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_first_event_wpa(top_tower)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_first_event_wpa(mid_tower)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_first_event_wpa(bot_tower)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
item_ids = [3047, 3111, 3158, 3006, 3009, 3020]

for item_id in item_ids:
    wr, we = calc_event_wpa(get_item_callback(item_id))
    mean, me = calculate_wpa_ci(wr, we)
    item_name = item_data['data'][str(item_id)]['name']
    print(f'{item_name}: ' + format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_first_event_wpa(baron)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
wr, we = calc_average_event_wpa(champion_kill)
mean, me = calculate_wpa_ci(wr, we)
print(format_wpa_summary(mean, me))

In [ ]:
len(all_files)

In [ ]:
# Adds training path to all_files
for dirpath, dirnames, filenames in os.walk('../dataset/train'):
    for filename in filenames:
        all_files.append(os.path.join(dirpath, filename))

In [ ]:
len(all_files)

In [ ]:
records = []

for path in all_files:
    with open(path) as f:
        game = json.load(f)
        results = sample_every(game, get_item_callback(2055))
        blue_wards = 0
        red_wards = 0
        for state, outcome in results:
            if outcome:
                blue_wards += 1
            else:
                red_wards += 1
            
        records.append((blue_wards, state['win']))
        records.append((red_wards, not state['win']))

In [ ]:
len(records)

In [ ]:
df = pd.DataFrame(records, columns=["value", "outcome"])
df["outcome"] = df["outcome"].astype(int)

In [ ]:
bin_width = 3
bins = range(0, df["value"].max() + bin_width, bin_width)

df["bin"] = pd.cut(df["value"], bins=bins, right=False)

In [ ]:
sns.set_theme(style='darkgrid', context='notebook')

In [ ]:
max_value = 20
filtered = df[df['value'] <= max_value]

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=filtered,
    x="value",
    y="outcome",
    estimator="mean",
    errorbar=("ci", 95)
)
plt.title('Winrate per Control Wards Purchased')
plt.xlabel('# of Control Wards')
plt.ylabel('Winrate')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.show()

In [ ]:
def get_game_paths(rank):
    game_paths = []
    if rank == 'MASTER':
        files = os.listdir('../dataset/data/MASTER')
        for f in files:
            game_paths.append(f'../dataset/data/MASTER/{f}')
    else:
        for tier in ['I', 'II', 'III', 'IV']:
            files = os.listdir(f'../dataset/data/{rank}/{tier}')
            for f in files:
                game_paths.append(f'../dataset/data/{rank}/{tier}/{f}')
    return game_paths

In [ ]:
all_game_paths = []

for rank in ['GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER']:
    all_game_paths += get_game_paths(rank)

In [ ]:
random.shuffle(all_game_paths)

In [ ]:
def team_with_more(attribute, frame):
    blue = 0
    red = 0
    if attribute == 'minions':
        for i in range(1, 6):
            participant = frame['participantFrames'][str(i)]
            cs = participant['minionsKilled'] + participant['jungleMinionsKilled']
            blue += cs
        for i in range(5, 11):
            participant = frame['participantFrames'][str(i)]
            cs = participant['minionsKilled'] + participant['jungleMinionsKilled']
            red += cs
    elif attribute == 'gold':
        for i in range(1, 6):
            blue += frame['participantFrames'][str(i)]['totalGold']
        for i in range(5, 11):
            red += frame['participantFrames'][str(i)]['totalGold']
    return blue > red

In [ ]:
def get_stat_winrates(stat):
    wins = [0] * 35
    total = [0] * 35

    if stat == 'kda':
        for path in all_files[:10]:
            with open(path) as f:
                game = json.load(f)
            state = sample(game, len(game['events']) - 1)
            blue_score = [0, 0, 0]
            red_score = [0, 0, 0]
            for p in state['teams'][0]['players']:
                blue_score[0] += p['kills']
                blue_score[1] += p['deaths']
                blue_score[2] += p['assists']
            for p in state['teams'][1]['players']:
                red_score[0] += p['kills']
                red_score[1] += p['deaths']
                red_score[2] += p['assists']
            blue_kda = (blue_score[0] + blue_score[2]) / max(blue_score[1], 1)
            red_kda = (red_score[0] + red_score[2]) / max(red_score[1], 1)
            blue_better = blue_kda > red_kda
            wins[
    else:
        for path in all_game_paths[:5000]:
            with open(path) as f:
                game = json.load(f)
                if game['timeline']['endOfGameResult'] != 'GameComplete':
                    continue
            blue_win = game['timeline']['frames'][-1]['events'][-1]['winningTeam'] == 100
            for frame_idx in range(min(len(game['timeline']['frames']), 35)):
                frame = game['timeline']['frames'][frame_idx]
                wins[frame_idx] += blue_win == team_with_more(stat, frame)
                total[frame_idx] += 1
    winrates = np.array(wins) / np.array(total)
    return winrates

In [ ]:
more_cs_wr = get_stat_winrates('minions')
more_gold_wr = get_stat_winrates('gold')

In [ ]:
get_stat_winrates('kda')